# Notebook 10 — Autonomy & Epistemic Transparency

This notebook demonstrates two new core pillars of the **Multigen** framework:

| Pillar | What it means |
|--------|---------------|
| **Dynamic Agent Self-Healing** | The workflow creates agents on demand, uses them, then destroys them — no manual registration required |
| **Epistemic Transparency** | Every node reports *what it knows*, *how confident it is*, *what it assumed*, and *what it doesn't know* |

Together these two features let Multigen operate autonomously at the frontier of its knowledge — while being explicit and honest about its limitations.

---

**Workflow used in this demo:**
```
data_prep (EchoAgent)
    → financial_model (QuantitativeMacroEconomistAgent ← DOES NOT EXIST → triggers approval gate)
        → synthesis (EchoAgent)
```


In [ ]:
import json, time
from multigen import SyncMultigenClient, GraphBuilder
from multigen.models import FanOutNodeDef, FanOutRequest

ORCHESTRATOR_URL = "http://localhost:8009"
client = SyncMultigenClient(base_url=ORCHESTRATOR_URL, timeout=60)
assert client.ping()
print("Connected")


---

## Section 1 — Epistemic Transparency Explained

### What is epistemic transparency?

In most AI pipelines, nodes return a result — and that's all you get. You have no idea:
- Was the agent confident or guessing?
- What data was missing?
- What assumptions did it make silently?
- What is it structurally incapable of assessing?

**Epistemic transparency** changes this. Every agent output includes a structured `epistemic` dictionary alongside its primary output:


### The `epistemic` schema

| Field | Type | Description |
|-------|------|-------------|
| `confidence` | `float` 0–1 | Overall confidence in the output |
| `reasoning` | `str` | Explanation of how the conclusion was reached |
| `uncertainty_sources` | `list[str]` | Which inputs were unclear, missing, or contradictory |
| `assumptions` | `list[str]` | What was assumed when data was absent |
| `known_limitations` | `list[str]` | What this agent structurally *cannot* assess |
| `known_unknowns` | `list[str]` | Identified gaps — things the agent knows it doesn't know |
| `evidence_quality` | `str` | `high` / `medium` / `low` / `none` |
| `data_completeness` | `float` 0–1 | How complete the input data was |
| `flags` | `list[str]` | `needs_human_review`, `extrapolated`, `contradictory_inputs`, etc. |

### Uncertainty propagation

Critically, uncertainty **propagates through the graph**. If `node_A` has confidence 0.5, then `node_B` (which depends on `node_A`'s output) inherits that uncertainty — its effective trustworthiness is bounded by what it received upstream.

The orchestrator tracks `propagated_uncertainty` per node so you always know the full epistemic chain.


In [ ]:
sample_epistemic = {
    "confidence": 0.78,
    "reasoning": "Analysis based on 3-year revenue history with seasonal adjustments. DCF assumes stable WACC.",
    "uncertainty_sources": ["no audited financials provided", "market comparables from 2023 (stale)"],
    "assumptions": ["revenue growth of 8% based on sector median", "terminal growth rate 2.5%"],
    "known_limitations": ["cannot assess off-balance-sheet liabilities", "no access to cap table"],
    "known_unknowns": ["regulatory filing status unknown", "customer concentration unreported"],
    "evidence_quality": "medium",
    "data_completeness": 0.72,
    "flags": [],
}
print(json.dumps(sample_epistemic, indent=2))


---

## Section 2 — Build a Graph with Unknown Agents

What happens when a node requests an agent that doesn't exist?

- **Traditional systems:** crash, `KeyError`, or silent skip
- **Multigen:** the system automatically generates a spec (via LLM or template fallback), **pauses for human approval**, creates the agent, runs it, then cleans up

The graph below uses **both** a registered agent (`EchoAgent`) and an unregistered agent (`QuantitativeMacroEconomistAgent`). The latter will trigger the approval gate.


In [ ]:
graph_def = (
    GraphBuilder()
    .node("data_prep")
        .agent("EchoAgent")  # exists
        .params(task="prepare financial data for NovaSemi Technologies")
        .timeout(30)
        .done()
    .node("financial_model")
        .agent("QuantitativeMacroEconomistAgent")  # DOES NOT EXIST — will trigger approval gate
        .params(
            company="NovaSemi Technologies",
            task_description="Perform a quantitative macroeconomic sensitivity analysis: model how a 25bp Fed rate increase affects NovaSemi's cost of capital, DCF valuation, and customer demand across 3 scenarios (base, shock, severe). Return scenario outputs with confidence intervals.",
        )
        .reflect(threshold=0.80, max_rounds=2, critic="EchoAgent")
        .timeout(90)
        .done()
    .node("synthesis")
        .agent("EchoAgent")  # exists
        .params(task="synthesise financial model outputs into investment memo")
        .timeout(30)
        .done()
    .edge("data_prep", "financial_model")
    .edge("financial_model", "synthesis")
    .entry("data_prep")
    .max_cycles(2)
    .build()
)
print(f"Graph: {len(graph_def['nodes'])} nodes")
print(f"Unknown agents: QuantitativeMacroEconomistAgent (will trigger approval gate)")


In [ ]:
resp = client.run_graph(graph_def=graph_def, payload={
    "company": "NovaSemi Technologies",
    "analysis_date": "2026-03-20",
})
wf_id = resp.instance_id
print(f"Workflow launched: {wf_id}")
print(f"State API: {ORCHESTRATOR_URL}/workflows/{wf_id}/state")


---

## Section 3 — Wait for the Approval Gate

The workflow now executes `data_prep` (EchoAgent — instant), then encounters `QuantitativeMacroEconomistAgent` which is **not registered**.

It automatically:
1. Calls the LLM to generate a BlueprintAgent spec from the node's `task_description`
2. Pauses the workflow and stores the spec in the approval queue
3. Waits — the workflow is frozen until a human acts

Let's poll until the gate is active.


In [ ]:
print("Waiting for approval gate to activate...")
for _ in range(30):
    time.sleep(2)
    try:
        approvals = client.get_pending_approvals(wf_id)
        if approvals:
            print(f"\n  APPROVAL GATE ACTIVE — {len(approvals)} agent(s) awaiting review")
            break
        health = client.get_health(wf_id)
        print(f"  Nodes executed so far: {client.get_metrics(wf_id).nodes_executed}", end='\r')
    except Exception:
        pass


In [ ]:
approvals = client.get_pending_approvals(wf_id)
if approvals:
    spec = approvals[0]
    print("=" * 65)
    print(f"  AGENT SPEC AWAITING YOUR APPROVAL")
    print("=" * 65)
    for key, val in spec.items():
        if key in ('system_prompt', 'capability_description'):
            print(f"\n  {key}:")
            print(f"    {val[:300]}...")
        elif isinstance(val, list):
            print(f"\n  {key}:")
            for item in val:
                print(f"    • {item}")
        else:
            print(f"\n  {key}: {val}")
    print("\n" + "=" * 65)
    print("  Review the spec above. You can:")
    print("  • APPROVE: client.approve_agent(wf_id, spec)")
    print("  • REJECT:  client.reject_agent(wf_id, spec['agent_name'], reason='...')")
    print("  • MODIFY:  edit spec['system_prompt'] then approve")
else:
    print("No pending approvals (workflow may already be running or completed)")
    spec = None


---

## Section 4 — Human Approves (or Rejects) the Agent

This is the **human-in-the-loop** moment. You have full visibility into the auto-generated spec:

- **Inspect** the system prompt, capabilities, and tool list
- **Modify** the spec — add constraints, sharpen the prompt, remove tools
- **Approve** — the agent is created, the workflow resumes, and the agent is destroyed when the workflow ends
- **Reject** — the node is skipped and added to `dead_letters`; the workflow continues without it

Here we approve, with a small editorial addition to the system prompt.


In [ ]:
if spec:
    # Optional: add our own guidance to the system prompt
    spec["system_prompt"] += (
        "\n\nAdditional instruction from reviewer: "
        "Ensure all scenario outputs include a 95% confidence interval. "
        "Flag any assumption that has more than 20% impact on the result."
    )
    result = client.approve_agent(wf_id, spec)
    print(f"  ✓ Agent '{result['agent_name']}' approved")
    print(f"  The workflow will now create the agent and continue execution.")


In [ ]:
# Example: what rejection looks like (not executed against live workflow)
rejection_example = {
    "code": "POST /workflows/{id}/reject-agent",
    "body": {
        "agent_name": "QuantitativeMacroEconomistAgent",
        "reason": "Insufficient scope — please use ValuationAgent instead and inject a macro_sensitivity node"
    },
    "effect": "Node skipped → added to dead_letters → workflow continues without this node"
}
print("Rejection example:")
print(json.dumps(rejection_example, indent=2))


---

## Section 5 — Monitor the Dynamic Agent

After approval, Multigen:
1. Instantiates the `BlueprintAgent` with the approved spec
2. Registers it under `QuantitativeMacroEconomistAgent`
3. Executes the `financial_model` node
4. Runs reflection (critic: EchoAgent, threshold: 0.80, max 2 rounds)
5. On workflow completion, **automatically deregisters** the agent

The agent lifecycle is fully managed — no manual cleanup required.


In [ ]:
print("Waiting for workflow to complete after approval...")
for _ in range(60):
    time.sleep(3)
    try:
        da = client.get_dynamic_agents(wf_id)
        metrics = client.get_metrics(wf_id)
        print(f"  Nodes executed: {metrics.nodes_executed} | "
              f"Created: {da.get('created',[])} | "
              f"Pending: {da.get('pending_approval', [])}",
              end='\r')
        if metrics.nodes_executed >= 3:
            print(f"\n  ✓ Workflow complete!")
            break
    except Exception:
        pass

da = client.get_dynamic_agents(wf_id)
print("\n  Dynamic Agent Lifecycle Summary:")
print(f"    Created:  {da.get('created', [])}")
print(f"    Approved: {da.get('approved', [])}")
print(f"    Rejected: {da.get('rejected', {})}")
print(f"    Pending:  {da.get('pending_approval', [])}")
print("\n  (After workflow completion, created agents are automatically deregistered)")


---

## Section 6 — Epistemic Transparency Report

Now let's look at what the system **knows** — and what it **doesn't**.

The epistemic report aggregates the `epistemic` dict from every node's output and produces:
- Per-node confidence and propagated uncertainty
- A pool of all **known unknowns** across the graph
- **Epistemic debt** items — gaps with severity ratings
- Nodes flagged for human review
- An overall trustworthiness assessment and recommendation

This is your audit trail for AI-assisted decisions.


In [ ]:
report = client.get_epistemic_report(wf_id)
print("=" * 65)
print("  EPISTEMIC TRANSPARENCY REPORT")
print("=" * 65)

summary = report.get('summary', {})
print(f"\n  Nodes assessed:           {summary.get('nodes_assessed', 0)}")
print(f"  Average confidence:       {summary.get('avg_confidence', 0):.1%}")
print(f"  Minimum confidence:       {summary.get('min_confidence', 0):.1%}")
print(f"  Avg propagated uncertainty: {summary.get('avg_propagated_uncertainty', 0):.1%}")
print(f"  Epistemic debt items:     {summary.get('epistemic_debt_items', 0)}")
print(f"  Nodes flagged for review: {summary.get('nodes_flagged_for_human_review', 0)}")
print(f"  Weakest evidence quality: {summary.get('weakest_evidence_quality', 'N/A')}")
print(f"  Total known unknowns:     {summary.get('total_known_unknowns', 0)}")

print(f"\n  Overall trustworthiness:")
print(f"    {report.get('overall_trustworthiness', 'N/A')}")

print(f"\n  Recommendation:")
print(f"    {report.get('recommendation', 'N/A')}")


In [ ]:
print("\n  Per-Node Epistemic State:")
print(f"  {'Node':<30} {'Conf':>6} {'Prop.Unc':>10} {'Evidence':>10} {'Unknowns':>9}")
print("  " + "\u2500" * 68)

node_states = report.get('node_states', {})
for node_id, state in node_states.items():
    conf = state.get('confidence', 0)
    prop = state.get('propagated_uncertainty', 0)
    ev = state.get('evidence_quality', 'N/A')
    unk = len(state.get('known_unknowns', []))
    flag = " \u26a0" if conf < 0.6 else ""
    print(f"  {node_id:<30} {conf:>5.1%} {prop:>10.1%} {ev:>10} {unk:>8}{flag}")


In [ ]:
unknowns = report.get('known_unknowns_pool', [])
debt = report.get('epistemic_debt', [])
flags = report.get('human_review_flags', [])

if unknowns:
    print(f"\n  Known Unknowns Across Graph ({len(unknowns)} total):")
    for u in unknowns:
        print(f"    • {u}")

if flags:
    print(f"\n  Nodes Flagged for Human Review:")
    for f in flags:
        print(f"    • {f['node_id']} ({f['agent']}) — {f['reason']} (confidence={f['confidence']:.1%})")
        for u in f.get('known_unknowns', []):
            print(f"        known unknown: {u}")


---

## Section 7 — Uncertainty Propagation Visualization

Uncertainty from upstream nodes flows downstream. A node with low confidence doesn't just affect its own output — it degrades the trustworthiness of everything that depends on it.

The charts below show:
- **Left:** Per-node confidence (bars) vs propagated uncertainty (line) through the pipeline
- **Right:** Epistemic debt broken down by severity (critical / high / medium / low)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

node_states = report.get('node_states', {})
pipeline = ['data_prep', 'financial_model', 'synthesis']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Epistemic Transparency \u2014 Uncertainty Propagation', fontsize=13, fontweight='bold')

# Chart 1: confidence and propagated uncertainty per node
confidences = []
prop_uncertainties = []
labels = []
for nid in pipeline:
    if nid in node_states:
        s = node_states[nid]
        confidences.append(s.get('confidence', 0.5))
        prop_uncertainties.append(s.get('propagated_uncertainty', 0.0))
        labels.append(nid)

if labels:
    x = range(len(labels))
    bars = ax1.bar(x, confidences, color=['#4CAF50' if c >= 0.75 else '#FF9800' if c >= 0.6 else '#F44336' for c in confidences], alpha=0.8, label='Confidence')
    ax1.plot(x, prop_uncertainties, 'r--o', linewidth=2, markersize=8, label='Propagated Uncertainty')
    ax1.set_xticks(list(x))
    ax1.set_xticklabels(labels, rotation=15, ha='right')
    ax1.set_ylabel('Score (0-1)')
    ax1.set_title('Confidence vs Propagated Uncertainty')
    ax1.set_ylim(0, 1.1)
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    ax1.axhline(y=0.6, color='orange', linestyle=':', alpha=0.7, label='Review threshold')
else:
    ax1.text(0.5, 0.5, 'No completed nodes yet\n(run workflow first)', 
             ha='center', va='center', transform=ax1.transAxes)
    ax1.set_title('Confidence vs Propagated Uncertainty')

# Chart 2: epistemic debt breakdown
debt_items = report.get('epistemic_debt', [])
if debt_items:
    severity_counts = {}
    for d in debt_items:
        sev = d.get('severity', 'low')
        severity_counts[sev] = severity_counts.get(sev, 0) + 1
    
    colors_sev = {'critical': '#F44336', 'high': '#FF9800', 'medium': '#FFC107', 'low': '#4CAF50'}
    sevs = list(severity_counts.keys())
    counts = [severity_counts[s] for s in sevs]
    ax2.pie(counts, labels=[f"{s}\n({c})" for s, c in zip(sevs, counts)],
            colors=[colors_sev.get(s, '#9E9E9E') for s in sevs],
            autopct='%1.0f%%', startangle=90)
    ax2.set_title(f'Epistemic Debt by Severity\n({len(debt_items)} total items)')
else:
    ax2.text(0.5, 0.5, 'No epistemic debt\n(agents reported no unknowns\nor workflow not complete)',
             ha='center', va='center', transform=ax2.transAxes)
    ax2.set_title('Epistemic Debt by Severity')

plt.tight_layout()
plt.savefig('epistemic_report.png', dpi=150, bbox_inches='tight')
plt.show()
print("\u2713 Epistemic report saved to epistemic_report.png")


---

## Section 8 — Full Workflow State with Epistemic Metadata

The workflow's distributed state is stored in MongoDB. Each node's document now contains both its primary output and its `epistemic` metadata.

This means the full uncertainty picture is **durable** — you can audit it days or weeks after the workflow completed.


In [ ]:
state = client.get_state(wf_id)
print(f"\n  Full workflow state — {state.count} nodes in MongoDB")
print(f"  {'Node':<25} {'Output keys':<40} {'Confidence'}")
print("  " + "\u2500" * 75)
for node in state.nodes:
    output = node.output or {}
    keys = list(output.keys())[:5]
    ep = output.get('epistemic', {})
    conf = ep.get('confidence', output.get('confidence', '?'))
    print(f"  {node.node_id:<25} {str(keys):<40} {conf}")


---

## Section 9 — Summary

### What was demonstrated

**Dynamic Agent Self-Healing**
- Automatic agent spec generation via LLM (or template fallback) when an unknown agent is encountered
- Human approval gate: inspect spec → optionally edit → approve or reject
- Dynamic BlueprintAgent creation post-approval, wired into the running workflow
- Automatic agent cleanup (deregister) at workflow end — no orphan agents

**Epistemic Transparency**
- Per-node epistemic report: confidence, propagated uncertainty, known unknowns, evidence quality
- All uncertainty flows through the graph — downstream nodes know how reliable upstream was
- Epistemic debt tracking: gaps are named, rated by severity, and surfaced to humans
- Durable audit trail in MongoDB — the uncertainty picture persists long after the workflow

**Together**, these pillars allow Multigen workflows to:
- Operate beyond the boundary of pre-registered capabilities
- Keep humans informed and in control at every inflection point
- Be honest about what they know, don't know, and assumed


### Epistemic Transparency as a Design Principle

Most AI systems are built to produce answers. The pressure is always toward confidence — a system that hedges too much feels useless. But this creates a subtle and dangerous failure mode: **silent overconfidence**. The system is wrong, but it sounds right.

Epistemic transparency inverts this default. Every agent is required to:
- Report what it doesn't know, alongside what it does
- Distinguish between things it *can't* assess (structural limitations) and things it *didn't* receive (data gaps)
- Make its assumptions explicit, so they can be challenged
- Rate the quality of its own evidence

This is not just a technical feature — it is a design philosophy. It says that **an AI system's most important output is not its answer, but its uncertainty**. An answer without uncertainty metadata is incomplete. It cannot be safely acted on, audited, or appealed.

In high-stakes domains — financial analysis, medical reasoning, legal research, strategic planning — the difference between a confident wrong answer and an honest uncertain one can be enormous. Epistemic transparency makes AI systems safer not by making them smarter, but by making them more honest.

Multigen's approach treats uncertainty as a **first-class citizen of the workflow graph**: it is tracked, propagated, aggregated, and surfaced. The goal is a system where you can always answer the question: *"How much should I trust this output, and why?"

---
*Notebook 10 of the Multigen framework series.*
